# Vivir más sin ser más rico

**Trabajo Práctico Final · Visualización de la Información · ITBA**  
Javier Peral Belmont y Pilar Frutos · Noviembre de 2026

En 2024, Costa Rica vivía 2,1 años más que Estados Unidos con poco más de un tercio de su PIB por habitante. ¿Qué tan bien describe el ingreso las diferencias de longevidad entre países? Primero miramos el patrón y sus excepciones. Después miramos cómo se relaciona la longevidad con el acceso al agua potable y dónde se concentra la exposición a la contaminación del aire.

**En Colab**, elegí *Entorno de ejecución → Ejecutar todo* para descargar los datos de Data360 y generar los gráficos.


In [308]:
!pip -q install babel pycountry

import time

import altair as alt
import numpy as np
import pandas as pd
import pycountry
import requests
from babel import Locale

alt.data_transformers.disable_max_rows()
# Números con coma decimal y punto de miles en ejes y fichas
alt.renderers.set_embed_options(format_locale={"decimal": ",", "thousands": ".", "grouping": [3], "currency": ["$", ""]})

API = "https://data360api.worldbank.org/data360"
DESDE, HASTA = 2000, 2024


In [309]:
def pedir(metodo, ruta, intentos=6, **kwargs):
    """Llama a la API reintentando: a veces responde con el cuerpo vacío o corta la conexión."""
    for intento in range(intentos):
        try:
            r = requests.request(metodo, f"{API}/{ruta}", timeout=120, **kwargs)
            r.raise_for_status()
            return r.json()
        except (requests.RequestException, ValueError):
            if intento == intentos - 1:
                raise
            time.sleep(2 * (intento + 1))


# 1. Exploración de Data360

Consultamos el catálogo de Data360 del Banco Mundial y contamos los indicadores por base. Elegimos **World Development Indicators (WDI)** porque reúne series anuales por país de economía, población y salud.

In [310]:
respuesta = pedir("POST", "searchv2", json={
    "search": "*",
    "top": 0,
    "count": True,
    "filter": "type eq 'indicator'",
    "facets": ["series_description/database_id,count:1000"],
})

bases = (pd.DataFrame(respuesta["@search.facets"]["series_description/database_id"])
         .rename(columns={"value": "base", "count": "indicadores"})
         .sort_values("indicadores", ascending=False, ignore_index=True))

print("Bases de datos en Data360:", len(bases))
print("Indicadores en total:", bases["indicadores"].sum())
bases.head(15)


Bases de datos en Data360: 171
Indicadores en total: 9850


,base,indicadores
0,WB_WDI,1545
1,IMF_BOP,1146
2,WB_EDSTATS,1072
3,WB_ES,584
4,IMF_IFS,495
5,WB_GS,363
6,WB_FINDEX,280
7,IMF_GFS_SOO,218
8,BS_SGI,193
9,WB_HNP,192


# 2. Selección de indicadores y preparación de los datos

| Indicador de WDI | Código en Data360 | Unidad |
|---|---|---|
| Esperanza de vida al nacer | `WB_WDI_SP_DYN_LE00_IN` | Años |
| PIB por habitante (PPA) | `WB_WDI_NY_GDP_PCAP_PP_KD` | Dólares internacionales constantes de 2021 |
| Acceso básico a agua potable | `WB_WDI_SH_H2O_BASW_ZS` | Porcentaje de la población |
| Exposición media a partículas finas PM2,5 | `WB_WDI_EN_ATM_PM25_MC_M3` | Microgramos por metro cúbico (µg/m³) |

El acceso básico al agua se refiere al uso de una fuente mejorada a no más de 30 minutos de ida y vuelta. La exposición a PM2,5 es un promedio anual ponderado por población.


### Cómo llega una observación

Este ejemplo muestra el registro de exposición a PM2,5 de Argentina en 2023. La API identifica la economía con `REF_AREA`, el año con `TIME_PERIOD` y el valor con `OBS_VALUE`. Usamos los totales cuando vienen desagregados y cruzamos las series por economía y año.


In [ ]:
obs = pedir("GET", "data", params={
    "DATABASE_ID": "WB_WDI",
    "INDICATOR": "WB_WDI_EN_ATM_PM25_MC_M3",
    "REF_AREA": "ARG",
    "TIME_PERIOD": 2023,
})["value"][0]
pd.DataFrame([obs])[["REF_AREA", "TIME_PERIOD", "INDICATOR", "OBS_VALUE", "UNIT_MEASURE"]]


,REF_AREA,TIME_PERIOD,INDICATOR,OBS_VALUE,UNIT_MEASURE
0,ARG,2023,WB_WDI_EN_ATM_PM25_MC_M3,14.346595,MCG_M3


In [317]:
def traer_indicador(codigo, columna):
    """Descarga todas las páginas de un indicador y devuelve una tabla país–año."""
    filas, skip = [], 0
    while True:
        pagina = pedir("GET", "data", params={"DATABASE_ID": "WB_WDI", "INDICATOR": codigo, "top": 1000, "skip": skip})
        filas += pagina["value"]
        skip += 1000
        if skip >= pagina["count"] or not pagina["value"]:
            break
    d = pd.DataFrame(filas)
    for dim in ["SEX", "AGE", "URBANISATION"]:      # nos quedamos con el total si hay desagregaciones
        if dim in d and d[dim].nunique() > 1:
            d = d[d[dim] == "_T"]
    return pd.DataFrame({
        "iso3": d["REF_AREA"],
        "anio": pd.to_numeric(d["TIME_PERIOD"]).astype(int),
        columna: pd.to_numeric(d["OBS_VALUE"], errors="coerce"),
    }).dropna()


INDICADORES = {
    "vida": "WB_WDI_SP_DYN_LE00_IN",
    "pib": "WB_WDI_NY_GDP_PCAP_PP_KD",
    "agua": "WB_WDI_SH_H2O_BASW_ZS",
    "pm25": "WB_WDI_EN_ATM_PM25_MC_M3",
}

series = {clave: traer_indicador(codigo, clave) for clave, codigo in INDICADORES.items()}


La API incluye agregados como el total mundial. Los excluimos con la lista de economías del Banco Mundial, que también aporta la región y el grupo de ingreso de cada país.

In [318]:
jerarquia = pedir("GET", "portal/v1/hierarchy", params={"type": "country"})["data"]

REGIONES = {
    "East Asia & Pacific": "Asia oriental y Pacífico",
    "Europe & Central Asia": "Europa y Asia central",
    "Latin America & Caribbean": "América Latina y el Caribe",
    "Middle East, North Africa, Afghanistan & Pakistan": "Medio Oriente y norte de África",
    "North America": "América del Norte",
    "South Asia": "Asia meridional",
    "Eastern and Southern Africa": "África subsahariana",
    "Western and Central Africa": "África subsahariana",
}
INGRESOS = {"Low income": "Bajo", "Lower middle income": "Medio bajo", "Upper middle income": "Medio alto", "High income": "Alto"}

regiones = pd.DataFrame([{"iso3": c["id"], "nombre_en": c["name"], "region": REGIONES[g["name"]]}
                         for g in jerarquia["HCL_WB_REGIONS"] for c in g["countries"]])
ingresos = pd.DataFrame([{"iso3": c["id"], "ingreso": INGRESOS.get(g["name"])}
                         for g in jerarquia["HCL_WB_INCOME"] for c in g["countries"]])
paises = regiones.merge(ingresos, on="iso3", how="left")
paises["ingreso"] = paises["ingreso"].fillna("Sin clasificar")

nombres_es = Locale("es").territories
def nombre_es(fila):
    pais = pycountry.countries.get(alpha_3=fila.iso3)
    iso2 = pais.alpha_2 if pais else {"XKX": "XK"}.get(fila.iso3)
    return nombres_es.get(iso2, {"CHI": "Islas del Canal"}.get(fila.iso3, fila.nombre_en))

paises["pais"] = paises.apply(nombre_es, axis=1)

In [319]:
panel = None
for s in series.values():
    panel = s if panel is None else panel.merge(s, on=["iso3", "anio"], how="outer")

panel = (panel[panel["anio"].between(1990, HASTA)]
         .merge(paises[["iso3", "pais", "region", "ingreso"]], on="iso3", how="inner")
         .sort_values(["iso3", "anio"], ignore_index=True))

**Período y cobertura.** El gráfico de PIB usa 2024 y el de agua potable recorre 2000–2024. El mapa llega hasta el último año con datos de PM2,5 y esperanza de vida. La tabla cuenta las economías con ambos datos en años seleccionados.


In [ ]:
aire = panel.loc[
    panel["anio"].between(DESDE, HASTA), ["iso3", "pais", "anio", "pm25", "vida"]
].dropna(subset=["pm25", "vida"]).copy()
ultimo_anio_aire = int(aire["anio"].max())

cobertura_aire = aire.groupby("anio").agg(
    economias=("iso3", "nunique"),
    pm25_mediana=("pm25", "median"),
)
cobertura_aire.rename(columns={
    "economias": "Economías con ambos datos",
    "pm25_mediana": "Mediana de exposición a PM2,5 (µg/m³)",
}).reindex([DESDE, 2010, 2020, ultimo_anio_aire]).round(1)


,Economías con ambos datos,"Mediana de exposición a PM2,5 (µg/m³)"
anio,,
2000,200,22.8
2010,200,20.6
2020,200,17.8
2023,200,17.3


# 3. ¿Los países con mayor ingreso viven más?

La línea estima la esperanza de vida según el PIB por habitante. Cada duplicación del ingreso ocupa la misma distancia horizontal. El color marca la **brecha** entre la esperanza de vida observada y la estimada. Azul y rojo indican una diferencia de al menos 1,5 años por encima o por debajo. Amarillo indica que el país queda cerca de la línea.

Usá **Mostrar países**, **Región** o **Buscar país** para explorar la dispersión. Pasá el cursor sobre un punto para ver los valores.

Ajustamos una regresión lineal de la esperanza de vida al nacer sobre el logaritmo del PIB por habitante.

In [329]:
corte = panel[panel["anio"] == 2024].dropna(subset=["vida", "pib"]).copy()
b, a = np.polyfit(np.log(corte["pib"]), corte["vida"], 1)
corte["esperada"] = a + b * np.log(corte["pib"])
corte["brecha"] = corte["vida"] - corte["esperada"]
r2 = 1 - corte["brecha"].var() / corte["vida"].var()

In [328]:
import numpy as np

columnas = ['pais', 'region', 'pib', 'vida', 'esperada', 'brecha']
g1 = corte.copy()
g1 = g1.dropna(subset=columnas)
g1 = g1[g1['pib'] > 0].copy()

def categoria(brecha):
    if brecha >= 1.5:
        return 'Por encima'
    if brecha <= -1.5:
        return 'Por debajo'
    return 'Cerca del modelo'

g1['comparacion'] = g1['brecha'].apply(categoria)
g1['destacado'] = g1['pais'].isin([
    'Japón', 'Costa Rica', 'Estados Unidos',
    'Argentina', 'India', 'Nigeria'
])

curva = pd.DataFrame({'pib': np.geomspace(700, 150_000, 120)})
curva['vida'] = a + b * np.log(curva['pib'])

vista = alt.param(
    name='vista', value='Todos',
    bind=alt.binding_radio(
        options=['Todos', 'Por encima', 'Cerca del modelo', 'Por debajo'],
        name='Mostrar países: '
    )
)
region = alt.param(
    name='region', value='Todas',
    bind=alt.binding_select(
        options=['Todas'] + sorted(g1['region'].dropna().unique().tolist()),
        name='Región: '
    )
)
buscar = alt.param(
    name='buscar', value='',
    bind=alt.binding(input='search', placeholder='Ej.: Chile', name='Buscar país: ')
)

filtro = "(vista == 'Todos' || datum.comparacion == vista) && (region == 'Todas' || datum.region == region)"
coincide = "indexof(lower(datum.pais), lower(trim(buscar))) >= 0"
hay_busqueda = "length(trim(buscar)) > 0"

x = alt.X(
    'pib:Q', title='PIB por habitante (miles de dólares)',
    scale=alt.Scale(type='log', domain=[700, 150_000]),
    axis=alt.Axis(
        values=[1000, 2000, 5000, 10_000, 20_000, 50_000, 100_000],
        labelExpr="format(datum.value / 1000, '.0f')", grid=False
    )
)
y = alt.Y(
    'vida:Q', title='Esperanza de vida (años)',
    scale=alt.Scale(domain=[50, 88]),
    axis=alt.Axis(grid=True, gridColor='#f3f3f3', gridWidth=0.7, tickCount=8)
)

linea = alt.Chart(curva).mark_line(color='#555555', strokeWidth=2).encode(x=x, y=y)
base = alt.Chart(g1).transform_filter(filtro)

puntos = base.mark_circle(size=75, stroke='white', strokeWidth=0.6).encode(
    x=x, y=y,
    color=alt.Color(
        'comparacion:N', title='Diferencia frente al modelo',
        scale=alt.Scale(
            domain=['Por encima', 'Cerca del modelo', 'Por debajo'],
            range=['#397ba7', '#d5b84f', '#b44d53']
        ),
        legend=alt.Legend(
            orient='top', direction='horizontal', columns=3,
            titleLimit=280, labelLimit=230, columnPadding=16,
            labelExpr=(
                "datum.label === 'Por encima' ? '1,5 años o más por encima' : "
                "datum.label === 'Por debajo' ? '1,5 años o más por debajo' : "
                "'Menos de 1,5 años de diferencia'"
            ),
        )
    ),
    opacity=alt.condition(
        f'!({hay_busqueda}) || ({coincide})',
        alt.value(0.83), alt.value(0.13)
    ),
    tooltip=[
        alt.Tooltip('pais:N', title='País'),
        alt.Tooltip('vida:Q', title='Esperanza de vida (años)', format='.1f'),
        alt.Tooltip('esperada:Q', title='Estimación del modelo (años)', format='.1f'),
        alt.Tooltip('brecha:Q', title='Diferencia (años)', format='+.1f'),
        alt.Tooltip('pib:Q', title='PIB por habitante (PPA)', format=',.0f')
    ]
)

etiquetas = base.transform_filter(
    f'(({hay_busqueda}) && ({coincide})) || (!({hay_busqueda}) && datum.destacado)'
).mark_text(
    align='left', dx=9, dy=-8, fontSize=11,
    fontWeight='bold', color='#292929'
).encode(x=x, y=y, text='pais:N')

dispersion_pib = (linea + puntos + etiquetas).properties(
    width=800, height=460,
    title=alt.Title(
        'Esperanza de vida y PIB por habitante, 2024. La línea muestra la estimación del modelo.',
        orient='bottom', anchor='start', fontSize=11, fontWeight='normal',
        color='#525252', offset=14,
    ),
)

grafico_1 = alt.vconcat(dispersion_pib).add_params(
    vista, region, buscar,
).properties(
    title=alt.Title(
        '¿Los países con mayor ingreso viven más?',
        orient='top', anchor='start', fontSize=17, offset=14,
    ),
).configure_view(stroke=None)

grafico_1


alt.VConcatChart(...)

En 2024, el modelo dio cuenta del 74,4 % de la variación en esperanza de vida entre 195 economías. Costa Rica y Estados Unidos muestran cuánto pueden diferir dos países del patrón promedio. Nauru ofrece otra pista para seguir la historia: vive casi 10 años menos de lo que estima el modelo. ¿Dónde aparece cuando comparamos la longevidad con el acceso al agua?

# 4. ¿Cómo se relaciona el acceso al agua con la esperanza de vida?

El color identifica la región del Banco Mundial. Primero se ve el panorama general y después el detalle de 90 % a 100 %. Cada gráfico tiene su propio deslizador y buscador. El gráfico general muestra el recorrido del país buscado.


In [332]:
def datos_anuales(col, decimales):
    d = panel[panel["anio"].between(DESDE, HASTA) & panel[col].notna() & panel["vida"].notna()]
    return d.assign(valor=d[col].round(decimales), vida=d["vida"].round(1))[["iso3", "pais", "region", "ingreso", "anio", "valor", "vida"]]


agua = datos_anuales("agua", 1)


In [333]:
# Cuenta las economías que aparecen en el año elegido.
cobertura_agua = agua.groupby("anio", as_index=False)["iso3"].nunique()
cobertura_agua["texto"] = cobertura_agua.apply(
    lambda fila: f'{fila["anio"]} · {fila["iso3"]} países y territorios con datos de agua y esperanza de vida',
    axis=1,
)

ANCHO, ALTO = 760, 460
anio_agua = alt.param(
    name="anio_agua",
    value=HASTA,
    bind=alt.binding_range(min=DESDE, max=HASTA, step=1, name="Año: "),
)
buscar_agua = alt.param(name="buscar_agua", value="", bind=alt.binding(input="search", placeholder="Ej.: Argentina", name="Buscar un país: "))
COINCIDE_AGUA = "length(buscar_agua) > 1 && indexof(lower(datum.pais), lower(buscar_agua)) >= 0"

escala_agua = alt.Scale(type="linear", domain=[0, 100], nice=False)
eje_agua = alt.Axis(values=list(range(0, 101, 10)), labelExpr="datum.value + ' %'")
x_agua = alt.X("valor:Q", title="Población con acceso básico al agua potable (%)", scale=escala_agua, axis=eje_agua)
vida_min_agua = int(agua["vida"].min() // 10) * 10
y_agua = alt.Y(
    "vida:Q", title="Esperanza de vida (años)",
    scale=alt.Scale(domain=[vida_min_agua, 88], nice=False),
)

base_agua = alt.Chart(agua).transform_filter(alt.datum.anio == anio_agua)
FICHA_AGUA = [
        alt.Tooltip("pais:N", title="País"),
        alt.Tooltip("region:N", title="Región"),
        alt.Tooltip("anio:Q", title="Año", format="d"),
        alt.Tooltip("valor:Q", title="Acceso básico a agua (% de la población)", format=".1f"),
        alt.Tooltip("vida:Q", title="Esperanza de vida (años)", format=".1f"),
]
COLORES_REGION_AGUA = {
    "África subsahariana": "#e54818",
    "América Latina y el Caribe": "#da286c",
    "América del Norte": "#2663db",
    "Asia meridional": "#519518",
    "Asia oriental y Pacífico": "#009a9d",
    "Europa y Asia central": "#8243db",
    "Medio Oriente y norte de África": "#ba7b00",
}
color_region_agua = alt.Color(
    "region:N",
    title="Región (Banco Mundial)",
    scale=alt.Scale(domain=list(COLORES_REGION_AGUA), range=list(COLORES_REGION_AGUA.values())),
    legend=alt.Legend(orient="right", symbolSize=75, symbolOpacity=1, labelLimit=230),
)
opacidad_puntos = (
    alt.when(COINCIDE_AGUA).then(alt.value(1))
    .when("length(buscar_agua) > 1").then(alt.value(0.18))
    .otherwise(alt.value(0.76))
)
puntos_agua = base_agua.mark_circle(size=55, stroke="white", strokeWidth=0.45).encode(
    x=x_agua, y=y_agua, color=color_region_agua, opacity=opacidad_puntos, tooltip=FICHA_AGUA,
)
# Recorrido del país buscado desde 2000 hasta el año elegido
recorrido = alt.Chart(agua).transform_filter(f"{COINCIDE_AGUA} && datum.anio <= anio_agua")
linea_recorrido = recorrido.mark_line(color="#262626", strokeWidth=1.5, point=alt.OverlayMarkDef(size=12, color="#262626")).encode(
    x=x_agua, y=y_agua, detail="pais:N", order="anio:Q")
nombre_recorrido = base_agua.transform_filter(COINCIDE_AGUA).mark_text(align="right", dx=-8, dy=-8, fontWeight="bold").encode(
    x=x_agua, y=y_agua, text="pais:N")

encabezado_agua = alt.Chart(cobertura_agua).transform_filter(
    alt.datum.anio == anio_agua
).mark_text(
    align="left", x=0, y=-14, fontSize=12, color="#444444",
).encode(text="texto:N")

grafico_2 = (
    puntos_agua + linea_recorrido + nombre_recorrido + encabezado_agua
).add_params(
    anio_agua, buscar_agua,
).properties(
    width=ANCHO, height=ALTO,
    padding={"top": 20, "left": 10, "right": 16, "bottom": 10},
    title=alt.Title(
        "¿Viven más los países donde más gente tiene agua potable?",
        anchor="start", fontSize=21, offset=30,
    ),
)
grafico_2


alt.LayerChart(...)

In [ ]:
# Detalle de países con entre 90 % y 100 % de acceso.
x_detalle_agua = alt.X(
    "valor:Q", title="Población con acceso básico al agua potable (%)",
    scale=alt.Scale(type="linear", domain=[90, 100], nice=False),
    axis=alt.Axis(values=[90, 92, 94, 96, 98, 100], labelExpr="datum.value + ' %'"),
)
acceso_alto = (alt.datum.valor >= 90) & (alt.datum.valor <= 100)
puntos_detalle_agua = puntos_agua.transform_filter(acceso_alto).encode(x=x_detalle_agua)
nombre_detalle_agua = nombre_recorrido.transform_filter(acceso_alto).encode(x=x_detalle_agua)

detalle_agua = (puntos_detalle_agua + nombre_detalle_agua).add_params(
    anio_agua, buscar_agua,
).properties(
    width=ANCHO, height=280,
    padding={"top": 16, "left": 10, "right": 16, "bottom": 10},
    title=alt.Title(
        "Detalle · Países con al menos 90 % de acceso al agua potable",
        anchor="start", fontSize=16, offset=12,
    ),
)
detalle_agua


alt.LayerChart(...)

Entre 2000 y 2024, la diferencia de esperanza de vida mediana entre los países con menos del 80 % y los que alcanzaban al menos el 95 % de acceso bajó de unos 21 a 14,5 años. En 2024, entre los países con al menos 95 % de acceso, la esperanza de vida iba de 62 a 86 años. Nauru queda en el extremo bajo, con casi 98 % de acceso y 62 años de esperanza de vida.

Entre países con acceso casi universal al agua potable, la esperanza de vida varía mucho. ¿Vemos algo parecido entre países con niveles similares de contaminación del aire?


# 5. ¿Cómo se relacionan la contaminación del aire y la esperanza de vida?

El mapa muestra la exposición a PM2,5. A la derecha, cada punto cruza esa exposición con la esperanza de vida del mismo país y año. El deslizador cambia ambas vistas y un clic destaca el país en las dos. Hacé clic en el fondo para volver al panorama general. Las áreas grises quedan sin dato en el mapa.


In [335]:
codigos_mapa = {pais.alpha_3: pais.numeric for pais in pycountry.countries}
datos_mapa_aire = aire.assign(id=aire["iso3"].map(codigos_mapa)).dropna(subset=["id"])
mundo = alt.topo_feature(
    "https://cdn.jsdelivr.net/npm/world-atlas@2.0.2/countries-110m.json",
    "countries",
)


In [ ]:
anio_aire = alt.param(
    name="anio_aire",
    value=ultimo_anio_aire,
    bind=alt.binding_range(min=DESDE, max=ultimo_anio_aire, step=1, name="Año: "),
)
clic_en_pais_aire = "event.item && event.item.datum && isValid(event.item.datum.iso3)"
pais_aire = alt.selection_point(
    name="pais_aire", fields=["iso3"], toggle=False,
    on=f"click[{clic_en_pais_aire}]", clear=f"click[!({clic_en_pais_aire})]",
)
color_aire = alt.Color(
    "pm25:Q", title="Exposición a PM2,5 (µg/m³)",
    scale=alt.Scale(domain=[0, aire["pm25"].max()], scheme="viridis", reverse=True, nice=False),
    legend=alt.Legend(orient="bottom", gradientLength=320, tickCount=6),
)
ficha_aire = [
    alt.Tooltip("pais:N", title="País o territorio"),
    alt.Tooltip("anio:Q", title="Año", format="d"),
    alt.Tooltip("pm25:Q", title="Exposición a PM2,5 (µg/m³)", format=".1f"),
    alt.Tooltip("vida:Q", title="Esperanza de vida al nacer (años)", format=".1f"),
]

mapa_base = alt.Chart(mundo).transform_filter(alt.datum.id != "010").mark_geoshape(
    fill="#e6e8eb", stroke="white", strokeWidth=0.45,
)
mapa_aire = alt.Chart(datos_mapa_aire).transform_filter(
    alt.datum.anio == anio_aire
).transform_lookup(
    lookup="id", from_=alt.LookupData(mundo, key="id"), as_="geometria",
).transform_filter(
    "isValid(datum.geometria)"
).mark_geoshape(stroke="white", strokeWidth=0.45, cursor="pointer").encode(
    shape=alt.Shape("geometria:G"),
    color=color_aire,
    opacity=alt.condition(pais_aire, alt.value(1), alt.value(0.22)),
    tooltip=ficha_aire,
).properties(name="paises_mapa_aire").add_params(pais_aire)
mapa = (mapa_base + mapa_aire).project(type="equalEarth").properties(
    width=465, height=290, title=alt.Title("Distribución geográfica", anchor="start"),
)

x_aire = alt.X(
    "pm25:Q", title="Exposición a PM2,5 (µg/m³)",
    scale=alt.Scale(domainMin=0), axis=alt.Axis(grid=False),
)
y_aire = alt.Y(
    "vida:Q", title="Esperanza de vida al nacer (años)",
    scale=alt.Scale(domain=[10, 90], nice=False), axis=alt.Axis(gridColor="#ededed"),
)
base_aire = alt.Chart(aire).transform_filter(alt.datum.anio == anio_aire).encode(
    x=x_aire, y=y_aire,
)
puntos_aire = base_aire.mark_circle(size=48, stroke="white", strokeWidth=0.5, cursor="pointer").encode(
    color=color_aire,
    opacity=alt.condition(pais_aire, alt.value(0.9), alt.value(0.16)),
    tooltip=ficha_aire,
).properties(name="paises_dispersion_aire").add_params(pais_aire)

# Ancla el eje horizontal con el máximo de todo el período.
limite_aire = alt.Chart(aire).transform_aggregate(pm25="max(pm25)").mark_point(opacity=0).encode(x=x_aire)
dispersion_aire = (limite_aire + puntos_aire).properties(
    width=385, height=290, title=alt.Title("Exposición y esperanza de vida", anchor="start"),
)
grafico_3 = alt.hconcat(mapa, dispersion_aire, spacing=22).add_params(
    anio_aire,
).properties(
    padding={"top": 16, "left": 10, "right": 16, "bottom": 10},
    title=alt.Title("Contaminación del aire y esperanza de vida", anchor="start", fontSize=21, offset=12),
).configure_view(stroke=None)
grafico_3


alt.HConcatChart(...)

En 2023, **Bangladés y Nigeria tenían una exposición parecida a PM2,5** (69,3 y 65,7 µg/m³), pero en Bangladés se vivía unos **20 años más** (74,7 frente a 54,5).

En el conjunto de las 200 economías, la exposición a PM2,5 mostró una relación débil con la esperanza de vida.


# 6. Conclusión

En general, los países con mayor ingreso por habitante viven más. Aun así, en 2024 Costa Rica tenía dos años más de esperanza de vida que Estados Unidos con poco más de un tercio de su ingreso por habitante.

También encontramos grandes diferencias entre países con condiciones parecidas. En 2024, entre los que tenían acceso casi universal al agua potable, la esperanza de vida iba de 62 a 86 años. En 2023, Bangladés y Nigeria tenían niveles similares de contaminación del aire y unos 20 años de diferencia en esperanza de vida.

### Fuentes

- **Datos:** *World Development Indicators*, vía [Data360](https://data360.worldbank.org/en/api). La esperanza de vida proviene de estimaciones de Naciones Unidas y los datos de agua del Programa Conjunto de Monitoreo OMS/UNICEF.
- **Aire:** [Exposición anual a PM2,5](https://data.worldbank.org/indicator/EN.ATM.PM25.MC.M3), Banco Mundial.
- **Geometrías:** [World Atlas](https://github.com/topojson/world-atlas). A esta resolución, algunas economías pequeñas de la tabla de cobertura no tienen representación en el mapa.
